In [1]:
import csv
import json
import random
from datetime import date, timedelta

random.seed(42)

CLIENTE_ID = "CLI001"

# 1. transacoes.csv

* 6 meses de histórico: Março a Agosto/2026 (Agosto parcial, até dia 17 - "hoje")
* Categorias de despesa com valor-base mensal + variação normal (ruído pequeno)
* Alimentação recebe um aumento proposital nos últimos 2 meses para gerar o alerta de tendência usado na documentação

In [3]:
categorias_despesa = {
    "Alimentação": {
        "base": 650,
        "subcategorias": ["Supermercado", "Restaurante", "Delivery", "Padaria"],
        "forma_pagamento": ["Cartão de Crédito", "Pix", "Débito"],
    },
    "Transporte": {
        "base": 420,
        "subcategorias": ["Combustível", "Aplicativo", "Estacionamento", "Transporte Público"],
        "forma_pagamento": ["Cartão de Crédito", "Pix", "Débito"],
    },
    "Moradia": {
        "base": 1450,
        "subcategorias": ["Aluguel", "Condomínio", "Energia", "Água", "Internet"],
        "forma_pagamento": ["Débito Automático", "Pix"],
    },
    "Lazer": {
        "base": 280,
        "subcategorias": ["Cinema", "Streaming", "Viagem", "Show/Evento"],
        "forma_pagamento": ["Cartão de Crédito", "Pix"],
    },
    "Saúde": {
        "base": 310,
        "subcategorias": ["Plano de Saúde", "Farmácia", "Consulta"],
        "forma_pagamento": ["Débito Automático", "Cartão de Crédito"],
    },
    "Educação": {
        "base": 250,
        "subcategorias": ["Curso Online", "Mensalidade", "Material"],
        "forma_pagamento": ["Cartão de Crédito", "Débito Automático"],
    },
    "Compras": {
        "base": 300,
        "subcategorias": ["Vestuário", "Eletrônicos", "Casa"],
        "forma_pagamento": ["Cartão de Crédito", "Pix"],
    },
    "Assinaturas": {
        "base": 120,
        "subcategorias": ["Streaming", "Software", "Academia"],
        "forma_pagamento": ["Cartão de Crédito", "Débito Automático"],
    },
}

descricoes = {
    "Supermercado": ["Supermercado Pão de Açúcar", "Carrefour", "Extra Mercado"],
    "Restaurante": ["Restaurante Sabor Caseiro", "Outback", "Restaurante Japonês"],
    "Delivery": ["iFood", "Rappi"],
    "Padaria": ["Padaria Bom Pão", "Padaria Central"],
    "Combustível": ["Posto Ipiranga", "Posto Shell"],
    "Aplicativo": ["Uber", "99"],
    "Estacionamento": ["Estapar"],
    "Transporte Público": ["Bilhete Único"],
    "Aluguel": ["Aluguel Apartamento"],
    "Condomínio": ["Condomínio Edifício"],
    "Energia": ["Enel Energia"],
    "Água": ["Sabesp"],
    "Internet": ["Vivo Fibra"],
    "Cinema": ["Cinemark"],
    "Streaming": ["Netflix", "Spotify", "Amazon Prime"],
    "Viagem": ["Passagem Aérea", "Hotel"],
    "Show/Evento": ["Ingresso Show"],
    "Plano de Saúde": ["Bradesco Saúde"],
    "Farmácia": ["Droga Raia", "Drogasil"],
    "Consulta": ["Consulta Médica"],
    "Curso Online": ["Udemy", "Alura"],
    "Mensalidade": ["Mensalidade Curso"],
    "Material": ["Livraria Cultura"],
    "Vestuário": ["Renner", "Zara"],
    "Eletrônicos": ["Magazine Luiza"],
    "Casa": ["Leroy Merlin"],
    "Academia": ["Smart Fit"],
    "Software": ["Adobe Creative Cloud"],
}

meses = [
    (2026, 3), (2026, 4), (2026, 5), (2026, 6), (2026, 7), (2026, 8),
]

rows = []
tid = 1

# Salário (crédito) todo dia 5
for (ano, mes) in meses:
    if (ano, mes) == (2026, 8):
        # mês corrente ainda não fechou, mas salário já cai no início do mês
        pass
    rows.append({
        "id": f"T{tid:04d}",
        "data": date(ano, mes, 5).isoformat(),
        "categoria": "Salário",
        "subcategoria": "Salário Mensal",
        "descricao": "Crédito de Salário - Empregador",
        "valor": 6200.00,
        "tipo": "credito",
        "forma_pagamento": "Transferência",
        "cliente_id": CLIENTE_ID,
    })
    tid += 1

for cat, cfg in categorias_despesa.items():
    for (ano, mes) in meses:
        base = cfg["base"]

        # Tendência proposital: Alimentação sobe nos últimos 2 meses (Jul/Ago)
        if cat == "Alimentação" and (ano, mes) == (2026, 7):
            base = cfg["base"] * 1.18
        if cat == "Alimentação" and (ano, mes) == (2026, 8):
            base = cfg["base"] * 1.32

        # número de lançamentos no mês para essa categoria (parcelado em compras menores)
        n_lancamentos = random.randint(2, 5)
        valor_restante = base * random.uniform(0.9, 1.1)

        # Agosto é parcial (só até dia 17) -> menos lançamentos e valor proporcional
        dias_no_mes = 17 if (ano, mes) == (2026, 8) else 28

        for i in range(n_lancamentos):
            subcategoria = random.choice(cfg["subcategorias"])
            descricao = random.choice(descricoes[subcategoria])
            valor_lancamento = round(valor_restante / n_lancamentos * random.uniform(0.7, 1.3), 2)
            dia = random.randint(1, dias_no_mes)
            rows.append({
                "id": f"T{tid:04d}",
                "data": date(ano, mes, dia).isoformat(),
                "categoria": cat,
                "subcategoria": subcategoria,
                "descricao": descricao,
                "valor": valor_lancamento,
                "tipo": "debito",
                "forma_pagamento": random.choice(cfg["forma_pagamento"]),
                "cliente_id": CLIENTE_ID,
            })
            tid += 1

rows.sort(key=lambda r: r["data"])
# renumerar IDs em ordem cronológica
for i, r in enumerate(rows, start=1):
    r["id"] = f"T{i:04d}"

with open("transacoes.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "id", "data", "categoria", "subcategoria", "descricao",
        "valor", "tipo", "forma_pagamento", "cliente_id"
    ])
    writer.writeheader()
    for r in rows:
        writer.writerow(r)

print(f"transacoes.csv gerado com {len(rows)} linhas")

transacoes.csv gerado com 176 linhas


# 2. perfil_investidor.json

In [5]:
perfil = {
    "cliente_id": CLIENTE_ID,
    "nome": "Mariana Costa",
    "idade": 34,
    "profissao": "Analista de Marketing",
    "renda_mensal_estimada": 6200.00,
    "perfil_risco": "Moderado",
    "tolerancia_a_risco": "Média - aceita oscilação limitada em troca de rentabilidade maior que a poupança",
    "horizonte_investimento": "Médio prazo (1 a 3 anos)",
    "objetivos_financeiros": [
        "Formar reserva de emergência",
        "Trocar de carro em 2 anos",
        "Começar a investir em renda variável de forma gradual"
    ],
    "reserva_emergencia_atual": 3500.00,
    "reserva_emergencia_meta": 18000.00,
    "possui_investimentos_previos": True,
    "produtos_atuais": ["Poupança", "CDB Liquidez Diária"],
    "data_ultima_atualizacao_perfil": "2026-06-10"
}

with open("perfil_investidor.json", "w", encoding="utf-8") as f:
    json.dump(perfil, f, ensure_ascii=False, indent=2)

print("perfil_investidor.json gerado")

perfil_investidor.json gerado


# 3. produtos_financeiros.json

In [6]:
produtos = [
    {
        "id": "PROD001",
        "nome": "CDB Liquidez Diária",
        "categoria": "Renda Fixa",
        "descricao": "CDB com resgate a qualquer momento sem perda de rentabilidade, indicado para reserva de emergência.",
        "rentabilidade": "100% do CDI",
        "liquidez": "Diária",
        "valor_minimo": 100.00,
        "perfis_compativeis": ["Conservador", "Moderado", "Arrojado"],
        "indicado_para": ["Reserva de emergência", "Sobra de caixa no curto prazo"]
    },
    {
        "id": "PROD002",
        "nome": "CDB 24 meses",
        "categoria": "Renda Fixa",
        "descricao": "CDB com prazo de 24 meses e rentabilidade maior em troca de menor liquidez.",
        "rentabilidade": "115% do CDI",
        "liquidez": "No vencimento",
        "valor_minimo": 500.00,
        "perfis_compativeis": ["Moderado", "Arrojado"],
        "indicado_para": ["Objetivos de médio prazo", "Troca de bens em 1-3 anos"]
    },
    {
        "id": "PROD003",
        "nome": "Fundo Multimercado Bradesco",
        "categoria": "Fundo de Investimento",
        "descricao": "Fundo com estratégia diversificada entre renda fixa e variável, buscando retorno acima do CDI.",
        "rentabilidade": "Variável - meta CDI + 2% a.a.",
        "liquidez": "D+30",
        "valor_minimo": 1000.00,
        "perfis_compativeis": ["Moderado", "Arrojado"],
        "indicado_para": ["Diversificação", "Médio e longo prazo"]
    },
    {
        "id": "PROD004",
        "nome": "Tesouro Selic",
        "categoria": "Renda Fixa - Título Público",
        "descricao": "Título público pós-fixado atrelado à Selic, considerado um dos investimentos mais seguros do mercado.",
        "rentabilidade": "Selic",
        "liquidez": "D+1",
        "valor_minimo": 50.00,
        "perfis_compativeis": ["Conservador", "Moderado", "Arrojado"],
        "indicado_para": ["Reserva de emergência", "Perfil conservador"]
    },
    {
        "id": "PROD005",
        "nome": "Fundo de Ações Bradesco Small Caps",
        "categoria": "Renda Variável",
        "descricao": "Fundo que investe em ações de empresas de menor capitalização, com maior potencial de retorno e maior volatilidade.",
        "rentabilidade": "Variável - sujeito a oscilações de mercado",
        "liquidez": "D+4",
        "valor_minimo": 200.00,
        "perfis_compativeis": ["Arrojado"],
        "indicado_para": ["Longo prazo", "Diversificação em renda variável"]
    },
    {
        "id": "PROD006",
        "nome": "Cartão de Crédito Bradesco Elo Nanquim",
        "categoria": "Cartão de Crédito",
        "descricao": "Cartão com programa de pontos e cashback, sem anuidade no primeiro ano.",
        "rentabilidade": None,
        "liquidez": None,
        "valor_minimo": 0.00,
        "perfis_compativeis": ["Conservador", "Moderado", "Arrojado"],
        "indicado_para": ["Controle de gastos", "Acúmulo de pontos"]
    }
]

with open("produtos_financeiros.json", "w", encoding="utf-8") as f:
    json.dump(produtos, f, ensure_ascii=False, indent=2)

print("produtos_financeiros.json gerado")

produtos_financeiros.json gerado


# 4. historico_atendimento.csv

In [7]:
atendimentos = [
    {
        "id": "AT001",
        "data": "2026-04-14",
        "canal": "App",
        "assunto": "Dúvida sobre fatura do cartão",
        "resumo": "Cliente questionou lançamento não reconhecido na fatura; identificado como compra parcelada.",
        "satisfacao": 4,
    },
    {
        "id": "AT002",
        "data": "2026-05-22",
        "canal": "Chat BIA",
        "assunto": "Consulta de produtos de investimento",
        "resumo": "Cliente perguntou sobre opções de CDB com liquidez diária para reserva de emergência.",
        "satisfacao": 5,
    },
    {
        "id": "AT003",
        "data": "2026-06-30",
        "canal": "Agência",
        "assunto": "Atualização de perfil de investidor",
        "resumo": "Cliente refez o questionário de perfil de risco, atualizado de Conservador para Moderado.",
        "satisfacao": 5,
    },
    {
        "id": "AT004",
        "data": "2026-07-18",
        "canal": "Chat BIA",
        "assunto": "Dúvida sobre gastos do mês",
        "resumo": "Cliente perguntou por que o gasto com alimentação estava mais alto que o normal.",
        "satisfacao": 4,
    },
]

with open("historico_atendimento.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "data", "canal", "assunto", "resumo", "satisfacao"])
    writer.writeheader()
    for a in atendimentos:
        writer.writerow(a)

print("historico_atendimento.csv gerado")

historico_atendimento.csv gerado
